# 58 — Chuỗi thời gian

Notebook cuối nhóm trung cấp. Dữ liệu thị trường **là** chuỗi thời gian, nên
đây là chương dùng nhiều nhất trong thực tế — và là chương pandas 3.0 xoá nhiều
thứ nhất.

⚠️ **Sáu mã tần suất bị xoá hẳn.** `M`, `Q`, `A`, `Y`, `H`, `T`, `S` đều ném
`ValueError`. Code pandas 2.x có `resample("M")` sẽ gãy ngay dòng đầu.

Notebook gồm:

1. `DatetimeIndex` — và vì sao nên đặt nó làm index
2. Bảng đầy đủ mã tần suất bị xoá và mã thay thế
3. `resample` — `label`, `closed`, và chỗ hai tham số này đổi kết quả
4. `rolling` — `closed`, cửa sổ theo **thời gian** thay vì theo số dòng
5. Múi giờ: EOD không có, intraday có `+07:00`

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd

import finlens
from finlens_examples import ap_dung_theme, duong, hom_nay, lui_ngay

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

gia = client.eod.stock.ohlcv("HPG", start=lui_ngay(HOM_NAY, nam=3)).sort_values("date")
print(f"pandas {pd.__version__} · HPG {len(gia)} phiên · {gia['date'].min():%d/%m/%Y} → {gia['date'].max():%d/%m/%Y}")

pandas 3.0.5 · HPG 747 phiên · 14/08/2023 → 12/08/2026


## 1 · `DatetimeIndex` — đặt thời gian làm index

finlens trả `date` là một **cột**, không phải index. Đó là đúng cho dạng long
nhiều mã. Nhưng khi làm việc với **một** chuỗi, đưa nó lên index mở ra cả một
nhóm phương thức.

In [2]:
chuoi = gia.set_index("date")["close"]

print(f"type index: {type(chuoi.index).__name__}")
print(f"dtype     : {chuoi.index.dtype}")
print(f"freq      : {chuoi.index.freq}   ← None vì phiên giao dịch không đều (nghỉ lễ, cuối tuần)")

type index: DatetimeIndex
dtype     : datetime64[ns]
freq      : None   ← None vì phiên giao dịch không đều (nghỉ lễ, cuối tuần)


### Cắt theo chuỗi ngày — chỉ `DatetimeIndex` mới làm được

In [3]:
print(f"Cả năm 2025      : {len(chuoi['2025'])} phiên")
print(f"Tháng 3/2025     : {len(chuoi['2025-03'])} phiên")
print(f"Từ 3/2025 tới nay: {len(chuoi['2025-03':])} phiên")
print()
print("Ba dòng trên KHÔNG chạy được nếu date còn là cột thường.")

Cả năm 2025      : 249 phiên
Tháng 3/2025     : 21 phiên
Từ 3/2025 tới nay: 362 phiên

Ba dòng trên KHÔNG chạy được nếu date còn là cột thường.


### Bộ truy cập `.dt` khi date vẫn là cột

Không phải lúc nào cũng nên đưa lên index — với frame nhiều mã thì `.dt` là
cách đúng.

In [4]:
nhieu_ma = client.eod.stock.ohlcv(["HPG", "VCB", "FPT"], start=lui_ngay(HOM_NAY, nam=1))

co_lich = nhieu_ma.assign(
    nam=lambda d: d["date"].dt.year,
    thang=lambda d: d["date"].dt.month,
    quy=lambda d: d["date"].dt.quarter,
    thu=lambda d: d["date"].dt.day_name(),
    tuan_iso=lambda d: d["date"].dt.isocalendar().week,
)
print("Số phiên theo thứ trong tuần:")
print(co_lich["thu"].value_counts().to_string())
print()
print("→ không có thứ Bảy và Chủ nhật, đúng như kỳ vọng với dữ liệu chứng khoán")

Số phiên theo thứ trong tuần:
thu
Wednesday    156
Tuesday      153
Thursday     147
Friday       147
Monday       147

→ không có thứ Bảy và Chủ nhật, đúng như kỳ vọng với dữ liệu chứng khoán


## 2 · ⚠️ Mã tần suất pandas 3.0 đã xoá

Đây là bảng đáng in ra dán lên tường nếu bạn đang di trú từ pandas 2.x.

In [5]:
thu_nghiem = [
    ("D", "ngày lịch", True),
    ("B", "ngày làm việc", True),
    ("W", "tuần", True),
    ("M", "cuối tháng", False),
    ("ME", "cuối tháng", True),
    ("MS", "đầu tháng", True),
    ("Q", "cuối quý", False),
    ("QE", "cuối quý", True),
    ("A", "cuối năm", False),
    ("Y", "cuối năm", False),
    ("YE", "cuối năm", True),
    ("H", "giờ", False),
    ("h", "giờ", True),
    ("T", "phút", False),
    ("min", "phút", True),
    ("S", "giây", False),
    ("s", "giây", True),
]

ket = []
for ma, mo_ta, mong_doi in thu_nghiem:
    try:
        pd.date_range("2024-01-01", periods=2, freq=ma)
        trang_thai = "✓ dùng được"
    except ValueError:
        trang_thai = "✗ ĐÃ XOÁ"
    ket.append({"mã": ma, "nghĩa": mo_ta, "pandas 3.0": trang_thai})

bang_freq = pd.DataFrame(ket)
bang_freq

,mã,nghĩa,pandas 3.0
0,D,ngày lịch,✓ dùng được
1,B,ngày làm việc,✓ dùng được
2,W,tuần,✓ dùng được
3,M,cuối tháng,✗ ĐÃ XOÁ
4,ME,cuối tháng,✓ dùng được
5,MS,đầu tháng,✓ dùng được
6,Q,cuối quý,✗ ĐÃ XOÁ
7,QE,cuối quý,✓ dùng được
8,A,cuối năm,✗ ĐÃ XOÁ
9,Y,cuối năm,✗ ĐÃ XOÁ


**Bảng thay thế:**

| pandas 2.x | pandas 3.0 | Ghi nhớ |
|---|---|---|
| `"M"` | `"ME"` | **M**onth **E**nd — `M` một mình nhập nhằng với *minute* |
| `"Q"` | `"QE"` | **Q**uarter **E**nd |
| `"A"` / `"Y"` | `"YE"` | **Y**ear **E**nd |
| `"H"` | `"h"` | viết thường |
| `"T"` | `"min"` | viết đủ chữ |
| `"S"` | `"s"` | viết thường |

Quy tắc chung: **kỳ hạn từ ngày trở lên viết HOA và nêu rõ đầu/cuối kỳ; kỳ hạn
dưới ngày viết thường.**

In [6]:
print("Chú ý sự khác nhau giữa đầu kỳ và cuối kỳ:")
print(f"  freq='ME' → {[str(d.date()) for d in pd.date_range('2024-01-01', periods=3, freq='ME')]}")
print(f"  freq='MS' → {[str(d.date()) for d in pd.date_range('2024-01-01', periods=3, freq='MS')]}")

Chú ý sự khác nhau giữa đầu kỳ và cuối kỳ:
  freq='ME' → ['2024-01-31', '2024-02-29', '2024-03-31']
  freq='MS' → ['2024-01-01', '2024-02-01', '2024-03-01']


## 3 · `resample` — đổi tần suất

`resample` là `groupby` cho trục thời gian. Nó cần index là `DatetimeIndex`.

In [7]:
thang = chuoi.resample("ME").agg(["first", "max", "min", "last", "count"])
thang.columns = ["mở", "cao", "thấp", "đóng", "số phiên"]
print(f"{len(chuoi)} phiên ngày → {len(thang)} tháng")
thang.tail(5).round(2)

747 phiên ngày → 37 tháng


,mở,cao,thấp,đóng,số phiên
date,,,,,
2026-04-30,24.24,25.49,23.79,24.77,20
2026-05-31,24.64,24.86,23.82,24.00,20
2026-06-30,24.05,24.35,23.20,23.30,22
2026-07-31,23.45,23.45,20.35,21.70,23
2026-08-31,22.55,22.55,21.85,22.20,8


### Gộp OHLCV cho đúng

Mỗi cột cần một phép gộp khác nhau. Đây là chỗ `agg` với dict trả công.

In [8]:
ohlcv_thang = (
    gia.set_index("date")
    .resample("ME")
    .agg({"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"})
    .dropna()
)
print("OHLCV tháng — mỗi cột một phép gộp riêng:")
ohlcv_thang.tail(4).round(2)

OHLCV tháng — mỗi cột một phép gộp riêng:


,open,high,low,close,volume
date,,,,,
2026-05-31,24.77,25.00,23.41,24.0,571960472.0
2026-06-30,24.00,24.35,23.20,23.3,368432900.0
2026-07-31,23.40,23.65,20.10,21.7,547734300.0
2026-08-31,21.70,22.65,21.65,22.2,161718300.0


⚠️ **Đừng dùng `.mean()` cho cả frame OHLCV.** Giá mở cửa trung bình tháng
không phải giá mở cửa của tháng, và `high` trung bình không phải đỉnh tháng:

In [9]:
sai = gia.set_index("date").resample("ME").mean(numeric_only=True).tail(1)
dung = ohlcv_thang.tail(1)
print(f"resample().mean() → high = {sai['high'].iloc[0]:.2f}")
print(f"resample().max()  → high = {dung['high'].iloc[0]:.2f}   ← đỉnh thật của tháng")
print(f"Chênh lệch: {dung['high'].iloc[0] - sai['high'].iloc[0]:.2f} nghìn VND")

resample().mean() → high = 22.42
resample().max()  → high = 22.65   ← đỉnh thật của tháng
Chênh lệch: 0.23 nghìn VND


### `label` và `closed` — hai tham số đổi kết quả

`resample` gộp các mốc vào từng khoảng. Hai câu hỏi phải trả lời:

- **`closed`**: khoảng đóng ở đầu hay cuối? (mốc biên thuộc khoảng nào)
- **`label`**: dòng kết quả mang nhãn đầu khoảng hay cuối khoảng?

In [10]:
tuan = {}
for label in ("left", "right"):
    for closed in ("left", "right"):
        r = chuoi.resample("W", label=label, closed=closed).last()
        tuan[f"label={label}, closed={closed}"] = {
            "số tuần": len(r),
            "nhãn đầu": str(r.index[0].date()),
            "giá trị đầu": round(float(r.iloc[0]), 2),
        }
pd.DataFrame(tuan).T

,số tuần,nhãn đầu,giá trị đầu
"label=left, closed=left",157,2023-08-13,17.92
"label=left, closed=right",157,2023-08-13,17.92
"label=right, closed=left",157,2023-08-20,17.92
"label=right, closed=right",157,2023-08-20,17.92


⚠️ **Với dữ liệu tài chính, mặc định của `W` là `label="right", closed="right"`**
— tức nhãn là **Chủ nhật cuối tuần**, một ngày thị trường đóng cửa. Nếu bạn vẽ
biểu đồ hay ghép với dữ liệu khác, mốc đó không khớp với phiên nào cả.

In [11]:
mac_dinh = chuoi.resample("W").last()
print(f"Nhãn mặc định của resample('W'): {[str(d.date()) for d in mac_dinh.index[:3]]}")
print(f"Các ngày đó là thứ: {[d.day_name() for d in mac_dinh.index[:3]]}")
print()
print("Muốn nhãn là phiên giao dịch cuối tuần thật thì gộp rồi lấy chính mốc đó:")
tuan_that = (
    gia.assign(ngay=gia["date"])  # giữ date thành cột trước khi đưa lên index
    .set_index("date")
    .resample("W")
    .agg(ngay_that=("ngay", "max"), phien_cuoi=("close", "last"))
    .dropna()
)
print(tuan_that.tail(3).to_string())
print()
print(f"Nhãn index là {tuan_that.index[-1].day_name()}, "
      f"phiên thật là {tuan_that['ngay_that'].iloc[-1].day_name()}")

Nhãn mặc định của resample('W'): ['2023-08-20', '2023-08-27', '2023-09-03']
Các ngày đó là thứ: ['Sunday', 'Sunday', 'Sunday']

Muốn nhãn là phiên giao dịch cuối tuần thật thì gộp rồi lấy chính mốc đó:
            ngay_that  phien_cuoi
date                             
2026-08-02 2026-07-31        21.7
2026-08-09 2026-08-07        22.0
2026-08-16 2026-08-12        22.2

Nhãn index là Sunday, phiên thật là Wednesday


## 4 · `rolling` — cửa sổ trượt

Khác `resample` ở chỗ nó **không đổi số dòng**: mỗi dòng nhận một giá trị tính
trên N dòng gần nhất.

In [12]:
co_ma = gia.set_index("date").assign(
    ma20=lambda d: d["close"].rolling(20).mean(),
    ma20_min_1=lambda d: d["close"].rolling(20, min_periods=1).mean(),
    do_lech=lambda d: d["close"].rolling(20).std(),
)
print(f"rolling(20)               → {co_ma['ma20'].isna().sum()} ô NaN đầu chuỗi")
print(f"rolling(20, min_periods=1) → {co_ma['ma20_min_1'].isna().sum()} ô NaN")
print()
print("⚠️ min_periods=1 cho ra giá trị ngay từ dòng đầu — nhưng dòng đầu là trung")
print("   bình của ĐÚNG MỘT giá trị, không phải trung bình 20 phiên.")
print(f"   ma20_min_1 dòng đầu = {co_ma['ma20_min_1'].iloc[0]:.2f} = chính close dòng đầu ({co_ma['close'].iloc[0]:.2f})")

rolling(20)               → 19 ô NaN đầu chuỗi
rolling(20, min_periods=1) → 0 ô NaN

⚠️ min_periods=1 cho ra giá trị ngay từ dòng đầu — nhưng dòng đầu là trung
   bình của ĐÚNG MỘT giá trị, không phải trung bình 20 phiên.
   ma20_min_1 dòng đầu = 19.04 = chính close dòng đầu (19.04)


### ⚠️ `closed=` — chỗ sinh ra thiên lệch nhìn trước

Mặc định, cửa sổ **bao gồm dòng hiện tại**. Với trung bình động thì đúng. Với
"đỉnh N phiên trước" thì sai — vì giá hôm nay tự tính vào đỉnh của chính nó.

In [13]:
so_sanh = gia.set_index("date").assign(
    dinh_20_gom_hom_nay=lambda d: d["close"].rolling(20).max(),
    dinh_20_khong_gom=lambda d: d["close"].rolling(20, closed="left").max(),
)
so_sanh = so_sanh.assign(
    vuot_dinh_sai=lambda d: d["close"] > d["dinh_20_gom_hom_nay"],
    vuot_dinh_dung=lambda d: d["close"] > d["dinh_20_khong_gom"],
)
print(f"Tín hiệu 'vượt đỉnh 20 phiên':")
print(f"  closed mặc định → {so_sanh['vuot_dinh_sai'].sum()} lần   ← LUÔN bằng 0, vì close ≤ max(gồm chính nó)")
print(f"  closed='left'   → {so_sanh['vuot_dinh_dung'].sum()} lần   ← đúng")

Tín hiệu 'vượt đỉnh 20 phiên':
  closed mặc định → 0 lần   ← LUÔN bằng 0, vì close ≤ max(gồm chính nó)
  closed='left'   → 69 lần   ← đúng


Đây chính là chi tiết notebook `33` dùng khi dựng screener tín hiệu. Thiếu
`closed="left"` thì điều kiện "vượt đỉnh" **không bao giờ đúng**, và bạn sẽ
ngồi tìm lỗi trong logic thay vì trong tham số.

### Cửa sổ theo **thời gian** thay vì theo số dòng

`rolling(20)` là 20 *dòng*. `rolling("20D")` là 20 *ngày lịch* — số dòng trong
mỗi cửa sổ sẽ khác nhau vì có ngày nghỉ.

In [14]:
hai_kieu = gia.set_index("date").assign(
    theo_dong=lambda d: d["close"].rolling(20).mean(),
    theo_ngay=lambda d: d["close"].rolling("20D").mean(),
    so_diem_20d=lambda d: d["close"].rolling("20D").count(),
)
print(f"rolling(20)    — luôn 20 điểm")
print(f"rolling('20D') — số điểm mỗi cửa sổ: {hai_kieu['so_diem_20d'].min():.0f} tới {hai_kieu['so_diem_20d'].max():.0f}")
print()
print("Dùng cửa sổ thời gian khi khoảng cách giữa các mốc KHÔNG đều — dữ liệu tick,")
print("hoặc khi bạn muốn '20 ngày' theo nghĩa lịch chứ không phải 20 phiên.")

rolling(20)    — luôn 20 điểm
rolling('20D') — số điểm mỗi cửa sổ: 1 tới 15

Dùng cửa sổ thời gian khi khoảng cách giữa các mốc KHÔNG đều — dữ liệu tick,
hoặc khi bạn muốn '20 ngày' theo nghĩa lịch chứ không phải 20 phiên.


## 5 · `shift`, `diff`, `pct_change`

Ba phép nhìn sang dòng bên cạnh. Với **một** chuỗi thì đơn giản; với frame
nhiều mã thì **phải qua `groupby`** — bài học đã đo ba lần trong repo này.

In [15]:
mot_ma = gia.set_index("date")["close"]
print("Trên MỘT chuỗi:")
print(f"  shift(1)      → giá phiên trước: {mot_ma.shift(1).iloc[-1]:.2f}")
print(f"  diff()        → thay đổi tuyệt đối: {mot_ma.diff().iloc[-1]:+.2f}")
print(f"  pct_change()  → thay đổi tương đối: {mot_ma.pct_change().iloc[-1]:+.2%}")
print()
print(f"  shift(-1)     → nhìn về TƯƠNG LAI: {mot_ma.shift(-1).iloc[-2]:.2f}")
print("  ⚠️ shift âm là công cụ tính lợi suất tương lai cho event study,")
print("     và cũng là cách nhanh nhất để tạo thiên lệch nhìn trước trong backtest.")

Trên MỘT chuỗi:
  shift(1)      → giá phiên trước: 22.05
  diff()        → thay đổi tuyệt đối: +0.15
  pct_change()  → thay đổi tương đối: +0.68%

  shift(-1)     → nhìn về TƯƠNG LAI: 22.20
  ⚠️ shift âm là công cụ tính lợi suất tương lai cho event study,
     và cũng là cách nhanh nhất để tạo thiên lệch nhìn trước trong backtest.


## 6 · Múi giờ — EOD không có, intraday có

Đây là khác biệt thật trong dữ liệu finlens và nó sẽ cắn bạn khi ghép hai loại.

In [16]:
eod = client.eod.stock.ohlcv("HPG", start=lui_ngay(HOM_NAY, ngay=10))
NGAY = gia["date"].drop_duplicates().nlargest(3).iloc[-1].strftime("%Y-%m-%d")
intra = client.intraday.stock.ohlcv("HPG", interval="15min", start=NGAY, end=NGAY)

print(f"EOD      'date': {eod['date'].dtype}")
print(f"          mẫu  : {eod['date'].iloc[-1]}")
print()
print(f"Intraday 'time': {intra['time'].dtype}")
print(f"          mẫu  : {intra['time'].iloc[-1]}")
print()
print("→ EOD là ngày trần (naive), intraday có múi giờ Asia/Ho_Chi_Minh (+07:00)")

EOD      'date': datetime64[ns]
          mẫu  : 2026-08-12 00:00:00

Intraday 'time': datetime64[ns, Asia/Ho_Chi_Minh]
          mẫu  : 2026-08-10 14:45:00+07:00

→ EOD là ngày trần (naive), intraday có múi giờ Asia/Ho_Chi_Minh (+07:00)


⚠️ **So sánh một mốc có múi giờ với một mốc không có sẽ ném lỗi.**

In [17]:
try:
    _ = intra["time"].iloc[0] > eod["date"].iloc[0]
except TypeError as e:
    print(f"So sánh trực tiếp → TypeError: {str(e)[:90]}")

So sánh trực tiếp → TypeError: Cannot compare tz-naive and tz-aware timestamps


Hai cách xử lý, chọn theo ý định:

In [18]:
MUI_GIO = "Asia/Ho_Chi_Minh"

# Cách 1 — bỏ múi giờ khỏi intraday (khi bạn chỉ cần ngày)
bo_mui = intra["time"].dt.tz_localize(None)
print(f"Bỏ múi giờ : {bo_mui.iloc[0]}  ({bo_mui.dtype})")

# Cách 2 — gắn múi giờ cho EOD (khi bạn cần so mốc chính xác)
gan_mui = eod["date"].dt.tz_localize(MUI_GIO)
print(f"Gắn múi giờ: {gan_mui.iloc[0]}  ({gan_mui.dtype})")
print()
print(f"Giờ so được: {intra['time'].iloc[0] > gan_mui.iloc[0]}")

Bỏ múi giờ : 2026-08-10 09:15:00  (datetime64[ns])
Gắn múi giờ: 2026-08-03 00:00:00+07:00  (datetime64[ns, Asia/Ho_Chi_Minh])

Giờ so được: True


⚠️ **`tz_localize` và `tz_convert` là hai việc khác nhau.**

- `tz_localize` — *khai báo* mốc naive này thuộc múi giờ nào. Không đổi giờ.
- `tz_convert` — *đổi* mốc đã có múi giờ sang múi giờ khác. Đổi giờ.

In [19]:
naive = pd.Timestamp("2026-08-12 09:15:00")
print(f"Gốc (naive)                    : {naive}")
print(f"tz_localize('Asia/Ho_Chi_Minh'): {naive.tz_localize(MUI_GIO)}   ← giờ giữ nguyên, thêm +07:00")
print(f"rồi tz_convert('UTC')          : {naive.tz_localize(MUI_GIO).tz_convert('UTC')}   ← giờ lùi 7 tiếng")
print()
print("Nhầm hai cái này là lệch đúng 7 tiếng — đủ để một lệnh khớp lúc 14h")
print("nhảy sang ngày hôm trước.")

Gốc (naive)                    : 2026-08-12 09:15:00
tz_localize('Asia/Ho_Chi_Minh'): 2026-08-12 09:15:00+07:00   ← giờ giữ nguyên, thêm +07:00
rồi tz_convert('UTC')          : 2026-08-12 02:15:00+00:00   ← giờ lùi 7 tiếng

Nhầm hai cái này là lệch đúng 7 tiếng — đủ để một lệnh khớp lúc 14h
nhảy sang ngày hôm trước.


## 7 · `Period` — khi bạn muốn *kỳ* chứ không phải *mốc*

`Timestamp` là một điểm. `Period` là một khoảng. Với báo cáo tài chính theo quý
thì `Period` diễn đạt đúng hơn.

In [20]:
theo_quy = gia.assign(quy=lambda d: d["date"].dt.to_period("Q"))
print(f"date          : {gia['date'].iloc[0]}")
print(f"to_period('Q'): {theo_quy['quy'].iloc[0]}   ← một khoảng, không phải một điểm")
print()
tom_tat_quy = theo_quy.groupby("quy", observed=True).agg(
    so_phien=("close", "count"), gia_cuoi=("close", "last")
)
print(tom_tat_quy.tail(4).round(2).to_string())

date          : 2023-08-14 00:00:00
to_period('Q'): 2023Q3   ← một khoảng, không phải một điểm

        so_phien  gia_cuoi
quy                       
2025Q4        66     23.57
2026Q1        57     24.02
2026Q2        62     23.30
2026Q3        31     22.20


In [21]:
print("Period biết đầu và cuối của chính nó:")
q = theo_quy["quy"].iloc[-1]
print(f"  {q} → từ {q.start_time.date()} tới {q.end_time.date()}")
print()
print(f"Cộng trừ theo kỳ: {q} + 1 = {q + 1}")
print(f"Đổi ngược về mốc: {q.to_timestamp(how='end').date()}")

Period biết đầu và cuối của chính nó:
  2026Q3 → từ 2026-07-01 tới 2026-09-30

Cộng trừ theo kỳ: 2026Q3 + 1 = 2026Q4
Đổi ngược về mốc: 2026-09-30


## 8 · Ghép lại: biểu đồ đa tần suất

Cùng một chuỗi, ba tần suất — chỉ đọc được sau khi `resample` đúng.

In [22]:
ve = []
for freq, ten in [("D", "ngày"), ("W", "tuần"), ("ME", "tháng")]:
    r = chuoi.resample(freq).last().dropna()
    ve.append(pd.DataFrame({"date": r.index, "close": r.values, "tần suất": ten}))

duong(
    pd.concat(ve),
    x="date",
    y="close",
    theo="tần suất",
    tieu_de="HPG — cùng một chuỗi, ba tần suất",
    phu_de="resample('ME') · mã 'M' của pandas 2.x đã bị xoá",
    nhan_y="nghìn VND",
)

## Tổng kết

| Bạn cần | Viết |
|---|---|
| Gộp lên tần suất lớn hơn | `.resample("ME").agg({...})` |
| OHLCV theo tháng | `agg({"open":"first","high":"max","low":"min","close":"last","volume":"sum"})` |
| Trung bình động | `.rolling(20).mean()` |
| Đỉnh N phiên **trước** | `.rolling(20, closed="left").max()` |
| Cửa sổ theo ngày lịch | `.rolling("20D")` |
| Bỏ múi giờ | `.dt.tz_localize(None)` |
| Khai báo múi giờ | `.dt.tz_localize("Asia/Ho_Chi_Minh")` |

**Năm điều đáng nhớ:**

1. ⚠️ **`M`, `Q`, `A`, `Y`, `H`, `T`, `S` đã bị xoá.** Dùng `ME`, `QE`, `YE`,
   `h`, `min`, `s`. Đây là lỗi đầu tiên bạn gặp khi chạy code pandas 2.x.
2. **`resample("W")` mặc định gắn nhãn Chủ nhật** — một ngày không có phiên nào.
3. ⚠️ **`rolling(...).max()` gồm cả dòng hiện tại**, nên điều kiện "vượt đỉnh"
   không bao giờ đúng. Cần `closed="left"`.
4. **`min_periods=1` cho ra giá trị ngay dòng đầu** — nhưng đó là trung bình của
   một điểm, không phải trung bình 20 phiên. Nó che mất vùng warm-up.
5. **EOD không có múi giờ, intraday có `+07:00`.** So sánh trực tiếp ném
   `TypeError`; `tz_localize` khai báo, `tz_convert` đổi giờ — nhầm là lệch 7 tiếng.

---

**Hết nhóm trung cấp.** Bốn notebook `55`–`58` phủ những thứ bạn dùng hằng ngày:
gộp nhóm, đổi hình dạng, ghép, và chuỗi thời gian.

Notebook cuối cùng `59_di_tru_va_hieu_nang` — đang được viết — sẽ gom toàn bộ
những gì pandas 3.0 đã xoá thành một bảng tra, kèm cách bắt chúng trong test.